# Bengaluru House Price Prediction

## Project Overview

This project builds a machine learning model to predict residential property prices in Bengaluru using property characteristics such as location, total square feet, number of bathrooms, and number of bedrooms (BHK).

### Objectives
- Explore and clean the Bengaluru housing dataset
- Handle missing values and inconsistent square-footage values
- Engineer useful features such as BHK and price per square foot
- Reduce noise using domain-specific outlier detection
- Encode categorical location data
- Train and evaluate a Linear Regression model
- Create a reusable function for house-price prediction

### Tech Stack
Python · NumPy · Pandas · Matplotlib · Scikit-learn


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.linear_model import LinearRegression

matplotlib.rcParams["figure.figsize"] = (20, 10)


## 2. Load and Inspect the Dataset


In [ ]:
# The CSV file should be placed in the same directory as this notebook.
df1 = pd.read_csv("Bengaluru_House_Data.csv")

print(f"Dataset shape: {df1.shape}")
display(df1.head())


In [ ]:
df1.info()


In [ ]:
df1.isnull().sum()


## 3. Data Cleaning


In [ ]:
# Remove columns that are not used by the prediction model.
df2 = df1.drop(
    ["area_type", "society", "balcony", "availability"],
    axis="columns"
)

display(df2.head())
print("Missing values after column selection:")
display(df2.isnull().sum())


In [ ]:
# Remove rows containing missing values in the remaining model features.
df3 = df2.dropna().copy()

print(f"Shape after removing missing values: {df3.shape}")
display(df3.head())


## 4. Feature Engineering


In [ ]:
# Convert size such as "2 BHK" or "4 Bedroom" into a numeric BHK feature.
df3["bhk"] = df3["size"].apply(lambda x: int(x.split(" ")[0]))

display(df3[["size", "bhk"]].head())


In [ ]:
# Identify non-numeric total_sqft values, including ranges such as "2100-2850".
def is_float(value):
    try:
        float(value)
        return True
    except:
        return False

df3[~df3["total_sqft"].apply(is_float)][["total_sqft"]].head(10)


In [ ]:
# Convert square-footage ranges to their average value.
def convert_sqft_to_num(value):
    tokens = value.split("-")
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(value)
    except:
        return None

df4 = df3.copy()
df4["total_sqft"] = df4["total_sqft"].apply(convert_sqft_to_num)
df4 = df4.dropna(subset=["total_sqft"]).copy()

display(df4.head(3))


In [ ]:
# Price is stored in lakh Indian rupees.
# Convert it to price per square foot for outlier analysis.
df5 = df4.copy()
df5["price_per_sqft"] = df5["price"] * 100000 / df5["total_sqft"]

display(df5[["location", "total_sqft", "price", "price_per_sqft"]].head())


## 5. Location Preprocessing


In [ ]:
# Standardize location names by removing leading/trailing whitespace.
df5["location"] = df5["location"].apply(lambda x: x.strip())

location_stats = (
    df5.groupby("location")["location"]
    .agg("count")
    .sort_values(ascending=False)
)

print(f"Unique locations before grouping rare locations: {len(df5.location.unique())}")
print(f"Locations with 10 or fewer records: {len(location_stats[location_stats <= 10])}")


In [ ]:
# Group low-frequency locations into an "other" category.
location_stats_less_than_10 = location_stats[location_stats <= 10]

df5["location"] = df5["location"].apply(
    lambda x: "other" if x in location_stats_less_than_10 else x
)

print(f"Unique locations after grouping rare locations: {len(df5.location.unique())}")


## 6. Outlier Detection and Removal


In [ ]:
# Remove properties with unusually small area per BHK.
df6 = df5[~(df5.total_sqft / df5.bhk < 300)].copy()

print(f"Shape after BHK-area outlier removal: {df6.shape}")
display(df6.price_per_sqft.describe())


In [ ]:
# Remove price-per-square-foot outliers within each location.
def remove_pps_outliers(df):
    df_out = pd.DataFrame()

    for location, subdf in df.groupby("location"):
        mean = np.mean(subdf.price_per_sqft)
        std = np.std(subdf.price_per_sqft)

        reduced_df = subdf[
            (subdf.price_per_sqft > (mean - std)) &
            (subdf.price_per_sqft <= (mean + std))
        ]

        df_out = pd.concat([df_out, reduced_df], ignore_index=True)

    return df_out

df7 = remove_pps_outliers(df6)
print(f"Shape after price-per-square-foot outlier removal: {df7.shape}")


In [ ]:
def plot_scatter_chart(df, location):
    bhk2 = df[(df.location == location) & (df.bhk == 2)]
    bhk3 = df[(df.location == location) & (df.bhk == 3)]

    plt.figure(figsize=(12, 7))
    plt.scatter(
        bhk2.total_sqft, bhk2.price,
        label="2 BHK", s=50
    )
    plt.scatter(
        bhk3.total_sqft, bhk3.price,
        marker="+", label="3 BHK", s=50
    )
    plt.xlabel("Total Square Feet Area")
    plt.ylabel("Price (Lakh Indian Rupees)")
    plt.title(location)
    plt.legend()
    plt.show()

plot_scatter_chart(df7, "Rajaji Nagar")


In [ ]:
def remove_bhk_outliers(df):
    exclude_indices = np.array([])

    for location, location_df in df.groupby("location"):
        bhk_stats = {}

        for bhk, bhk_df in location_df.groupby("bhk"):
            bhk_stats[bhk] = {
                "mean": np.mean(bhk_df.price_per_sqft),
                "std": np.std(bhk_df.price_per_sqft),
                "count": bhk_df.shape[0]
            }

        for bhk, bhk_df in location_df.groupby("bhk"):
            stats = bhk_stats.get(bhk - 1)

            if stats and stats["count"] > 5:
                exclude_indices = np.append(
                    exclude_indices,
                    bhk_df[bhk_df.price_per_sqft < stats["mean"]].index.values
                )

    return df.drop(exclude_indices, axis="index")

df8 = remove_bhk_outliers(df7)
print(f"Shape after BHK-based outlier removal: {df8.shape}")


In [ ]:
plot_scatter_chart(df8, "Rajaji Nagar")
plot_scatter_chart(df8, "Hebbal")


In [ ]:
plt.figure(figsize=(12, 7))
plt.hist(df8.price_per_sqft, rwidth=0.8)
plt.xlabel("Price Per Square Foot")
plt.ylabel("Count")
plt.title("Distribution of Price Per Square Foot")
plt.show()


## 7. Bathroom Validation


In [ ]:
plt.figure(figsize=(12, 7))
plt.hist(df8.bath, rwidth=0.8)
plt.xlabel("Number of Bathrooms")
plt.ylabel("Count")
plt.title("Distribution of Bathrooms")
plt.show()

display(df8[df8.bath > 10])


In [ ]:
# Remove properties with an unusually high number of bathrooms
# relative to their number of bedrooms.
df9 = df8[df8.bath < df8.bhk + 2].copy()

print(f"Shape after bathroom outlier removal: {df9.shape}")


## 8. Prepare Features for Machine Learning


In [ ]:
# Remove columns that are no longer needed for modeling.
df10 = df9.drop(["size", "price_per_sqft"], axis="columns").copy()

# One-hot encode the location feature.
dummies = pd.get_dummies(df10.location)

df11 = pd.concat(
    [df10, dummies.drop("other", axis="columns")],
    axis="columns"
)

# Remove the original categorical location column.
df12 = df11.drop("location", axis="columns")

display(df12.head(3))
print(f"Final modeling dataset shape: {df12.shape}")


In [ ]:
X = df12.drop(["price"], axis="columns")
y = df12["price"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


## 9. Train-Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=10
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 10. Train the Linear Regression Model


In [ ]:
lr_clf = LinearRegression()
lr_clf.fit(X_train, y_train)

test_r2 = lr_clf.score(X_test, y_test)
print(f"Test R² Score: {test_r2:.4f}")


## 11. Cross-Validation


In [ ]:
cv = ShuffleSplit(
    n_splits=5,
    test_size=0.2,
    random_state=0
)

cv_scores = cross_val_score(
    LinearRegression(),
    X,
    y,
    cv=cv
)

print("Cross-validation scores:", np.round(cv_scores, 4))
print(f"Mean CV R² Score: {cv_scores.mean():.4f}")


## 12. House Price Prediction


In [ ]:
def predict_price(location, sqft, bath, bhk):
    """Predict house price in lakh Indian rupees."""

    if location not in X.columns:
        print(f"Location '{location}' was not found in the model features.")
        return None

    x = np.zeros(len(X.columns))

    # The first three model features are total_sqft, bath, and bhk.
    x[0] = sqft
    x[1] = bath
    x[2] = bhk

    loc_index = np.where(X.columns == location)[0][0]
    x[loc_index] = 1

    return lr_clf.predict([x])[0]


In [ ]:
# Example predictions
examples = [
    ("1st Phase JP Nagar", 1000, 2, 2),
    ("Indira Nagar", 1000, 2, 2),
    ("Kothanur", 1500, 3, 3),
]

for location, sqft, bath, bhk in examples:
    prediction = predict_price(location, sqft, bath, bhk)
    print(
        f"{location} | {sqft} sqft | {bath} bath | {bhk} BHK "
        f"-> Predicted price: ₹{prediction:.2f} lakh"
    )


## 13. Conclusion

The project demonstrates an end-to-end classical machine learning workflow for Bengaluru house-price prediction. The main steps include data cleaning, feature engineering, location-based preprocessing, domain-specific outlier removal, one-hot encoding, Linear Regression, cross-validation, and reusable price prediction.

### Future Improvements
- Compare Linear Regression with Ridge, Lasso, Random Forest, and Gradient Boosting
- Build a scikit-learn Pipeline for reproducible preprocessing and inference
- Add a web interface using Flask or Streamlit
- Save the trained model and expose it through an API
- Track additional evaluation metrics such as MAE and RMSE
